# Setup

In [ ]:
%run notebook_setup.py

import pandas as pd
import numpy as np
from scipy.stats import trim_mean
import polars as pl
import pandas as pd
import logging
import gc
import random

import matplotlib.pyplot as plt
import seaborn as sns

from src.config.dir_config import OUTPUT_PATH_DEMAND_SUMMARY
from src.config.bigquery_config import CREDENTIALS_GBQ, PROJECT_ID_GBQ
from src.utils.read_data import read_data
from src.utils.setup_logging import setup_logging
from src.utils.create_week_date import add_week_start_date
from src.utils.export_as_excel import export_dataframes_as_tables


import logging

setup_logging()

# Functions

In [ ]:
def add_real_sales_sv(df):
    def var_forward_sum(g):
        g = g.sort_values("week_number")
        sales = g["weekly_sales"].fillna(0).to_numpy(int)
        weeks = g["semana_vta"].fillna(1).astype(int).to_numpy()
        n = len(sales)

        s = np.cumsum(sales)
        i = np.arange(n)
        right = np.minimum(i + weeks, n)
        s_pad = np.concatenate(([0.0], s))
        
        return pd.Series(s_pad[right] - s_pad[i], index=g.index)

    df["real_sales"] = (
    df
    .groupby(["cod_sucursal", "cod_producto", "cod_talla"], group_keys=False)[["week_number", "weekly_sales", "semana_vta"]]
    .apply(var_forward_sum)
    .astype("int16")
    )

    return df

# Data import

In [ ]:
data_genex = pd.read_parquet('../data/processed/genex_data_processed.parquet')
demand_summary = pd.read_parquet('../data/processed/demand_summary.parquet')
data_sales = pd.read_parquet('../data/processed/weekly_sales_by_season/Invierno/weekly_sales_Invierno_2025_processed.parquet')
classifications = pd.read_excel('../data/external/Consolidado clasificaciones modelo BI.xlsx')
transfers = pd.read_parquet('../data/processed/trf_data_processed.parquet')

# Data processing

In [ ]:
data_sales = data_sales[data_sales['cod_sucursal'] != 767]
demand_summary = demand_summary[demand_summary['cod_sucursal'] != 767]
demand_summary = demand_summary[demand_summary['nombre_depto'] != 'Bolsas y bolsos']
demand_summary = demand_summary[demand_summary['nombre_depto'] != 'Miscelaneos']
demand_summary = demand_summary[demand_summary['nombre_temporada'] == 'Invierno']
demand_summary = demand_summary[demand_summary['ano_temporada'] == '2025']

In [ ]:
demand_info = demand_summary[['cod_producto', 'cod_talla', 'cod_sucursal',
                              'nombre_sucursal',
                              'nombre_temporada','ano_temporada','nombre_depto','nombre_linea','nom_talla',
                              'demand_type']].copy()

In [ ]:
transfers_pivot = transfers.pivot_table(
    index=['cod_sucursal','cod_producto','cod_talla','cod_ano_comercial','cod_semana'],
    observed=True,
    columns="nombre_razon_group",
    values= 'cantidad_des',
    aggfunc="sum",
    fill_value=0
).reset_index()

transfers_pivot.columns.name = None

In [ ]:
transfers_pivot_summary = transfers.pivot_table(
    index=['cod_sucursal','cod_producto','cod_talla'],
    observed=True,
    columns="nombre_razon_group",
    values= 'cantidad_des',
    aggfunc="sum",
    fill_value=0
).reset_index()

transfers_pivot_summary.columns.name = None

In [ ]:
demand_summary = demand_summary.merge(
    transfers_pivot_summary,
    on = ['cod_sucursal','cod_producto','cod_talla'],
    how = 'left'
)

demand_summary[["PREDISTRIBUIDA",
        "REPOSICION AUTOMATIC",
        "CARGA MANUAL",
        "OTRAS"]] = demand_summary[["PREDISTRIBUIDA",
                                    "REPOSICION AUTOMATIC",
                                    "CARGA MANUAL",
                                    "OTRAS"]].fillna(0)

## *A) Proccessing data_all*

In [ ]:
data_all = data_sales.merge(data_genex,
                            on=['cod_producto','cod_talla', 'cod_sucursal', 'cod_ano_comercial','cod_semana'],
                                how='left')

data_all = data_all.merge(demand_info,
                            on=['cod_producto','cod_talla', 'cod_sucursal'],
                            how='left')

data_all = data_all.merge(classifications,
                            on=['nombre_temporada','cod_sucursal','nombre_depto', 'nombre_linea'],
                            how='left')

data_all = data_all.merge(transfers_pivot,
                            on=['cod_producto','cod_talla', 'cod_sucursal', 'cod_ano_comercial','cod_semana'],
                                how='left')

del demand_info, classifications, data_sales
gc.collect()

In [ ]:
data_all = add_week_start_date(data_all)

In [ ]:
data_all['clasificacion'] = pd.Categorical(data_all['clasificacion'],
                                           categories=['AA','A','B','C'],
                                           ordered=True)

In [ ]:
data_all['factor_l_dias'] = np.where(
    data_all['mean_sales_past_4_weeks'] == 0,
    np.nan,
    (data_all['vta_promedio'] / data_all['mean_sales_past_4_weeks']).round(3)
)

In [ ]:
data_all[["PREDISTRIBUIDA",
        "REPOSICION AUTOMATIC",
        "CARGA MANUAL",
        "OTRAS"]] = data_all[["PREDISTRIBUIDA",
                                    "REPOSICION AUTOMATIC",
                                    "CARGA MANUAL",
                                    "OTRAS"]].fillna(0)

In [ ]:
data_all = add_real_sales_sv(data_all)

In [ ]:
data_all.to_parquet('../sandbox/data_genex_venta_transfer.parquet')

In [ ]:
demand_summary.to_parquet('../sandbox/demand_summary.parquet')

# EDA

In [ ]:
data_all = pd.read_parquet('../sandbox/data_genex_venta_transfer.parquet')

## Sample product

In [ ]:
cod_producto = 680279

data_sample = data_all[data_all['cod_producto'] == cod_producto].copy()

interest_columns = ['cod_sucursal','nombre_sucursal',
                    'nombre_depto','nombre_linea','cod_producto', 'cod_talla', 'nom_talla',
                    'date','week_number',
                    'mnt_precio_vigente','weekly_available_stock','weekly_sales',
                    'vta_promedio','factor','semana_vta', 'ume','repo_x_dda','repo_x_ume','can_original','can_final',
                    'real_sales', 'forecast_tricot',
                    'valida_formula_repo','estado','clasif',
                     "PREDISTRIBUIDA","REPOSICION AUTOMATIC","CARGA MANUAL","OTRAS",]


data_sample = data_sample[interest_columns].round(2).reset_index(drop=True)

: 

In [ ]:
demand_summary_sample = demand_summary[demand_summary['cod_producto'] == cod_producto].copy()

In [ ]:
dataframes = {
    f"{cod_producto}_sale": data_sample,
    f"{cod_producto}_summary":demand_summary_sample
    }

export_dataframes_as_tables(
    dataframes,
    f'../sandbox/{cod_producto}.xlsx'
)